### Project DRISHTI: Lunar South Pole Navigation & Hazard Pipeline
**Objective:** Map traversable terrain, identify Permanently Shadowed Regions (PSRs), and compute an optimal rover path to subsurface water-ice inside the Shoemaker Crater region using DFSAR, LOLA, and LRO illumination data.

---

#### Phase 0 & 1: Environment Initialization and Spatial Overlap Audit
**Scientific Rationale:** Before applying any gradient mathematics or morphological filters, we must mathematically guarantee that our Topography (DEM) and Illumination datasets geographically encompass our target Radar Swath (CPR/Ice Map). 

Warping a small regional tile into a large radar bounding box causes GDAL to pad the void space with `0.0` values. In planetary GIS, algorithms interpret `0.0` elevation or slope as perfectly safe, flat terrain, leading to catastrophic false-positive landing site selections (the "Ghost Pixel" problem). 

This cell initializes our strict `-9999.0` NoData environment and audits the bounding boxes of the raw PDS/PGDA files against the DFSAR radar swath to confirm 100% true spatial overlap.

In [1]:
import rasterio
import numpy as np
from scipy.ndimage import (distance_transform_edt, binary_dilation, 
                           label, maximum_filter, minimum_filter,
                           generic_filter)
import os

# --- GLOBAL CONSTANTS ---
NODATA = -9999.0
BASE_DIR = r"C:\DRISHTI_POC"


def spatial_overlap_audit_v2():
    print("=" * 70)
    print("PHASE 1: SPATIAL OVERLAP AUDIT (Raw Source vs Radar Swath)")
    print("=" * 70)
    
    # Files we are checking
    files = {
        "DFSAR_RADAR_SWATH": os.path.join(BASE_DIR, "WARPED_CPR.tif"),
        "ICE_TARGETS": os.path.join(BASE_DIR, "DRISHTI_FINAL_ICE_MAP.tif"),
        "NEW_LOLA_DEM (LBL)": os.path.join(BASE_DIR, "LDEM_85S_10M.LBL"),
        "NEW_AVG_ILLUM": os.path.join(BASE_DIR, "AVGVISIB_85S_060M_201608.tiff"),
        "NEW_PSR_MASK": os.path.join(BASE_DIR, "LPSR_85S_060M_201608.tiff")
    }
    
    radar_bounds = None
    
    for name, fpath in files.items():
        if not os.path.exists(fpath):
            print(f"\n[{name}] FILE NOT FOUND: Check path -> {fpath}")
            continue
            
        try:
            with rasterio.open(fpath) as src:
                b = src.bounds
                crs = src.crs
                
                print(f"\n{name}")
                print(f"  Bounds: Left={b.left:.1f}, Right={b.right:.1f}, Bottom={b.bottom:.1f}, Top={b.top:.1f}")
                print(f"  Size:   {src.width} x {src.height} pixels")
                print(f"  CRS:    {crs.data['proj'] if crs and 'proj' in crs.data else crs}")
                
                if name == "DFSAR_RADAR_SWATH":
                    radar_bounds = b
                elif radar_bounds:
                    # Check if the new dataset completely encompasses the radar swath
                    covers_x = (b.left <= radar_bounds.left) and (b.right >= radar_bounds.right)
                    covers_y = (b.bottom <= radar_bounds.bottom) and (b.top >= radar_bounds.top)
                    
                    if covers_x and covers_y:
                        print("  ✅ STATUS: Completely encompasses the DFSAR Radar Swath. Safe for warping.")
                    else:
                        print("  ❌ STATUS: Does NOT fully encompass the Radar Swath. Data gaps will occur.")
        except Exception as e:
            print(f"\n[{name}] ERROR reading file: {e}")

# Execute Phase 1
spatial_overlap_audit_v2()

PHASE 1: SPATIAL OVERLAP AUDIT (Raw Source vs Radar Swath)

DFSAR_RADAR_SWATH
  Bounds: Left=-27473.8, Right=127476.2, Bottom=12328.7, Top=90053.7
  Size:   6198 x 3109 pixels
  CRS:    stere

ICE_TARGETS
  Bounds: Left=-27473.8, Right=127476.2, Bottom=12328.7, Top=90053.7
  Size:   6198 x 3109 pixels
  CRS:    stere
  ✅ STATUS: Completely encompasses the DFSAR Radar Swath. Safe for warping.

NEW_LOLA_DEM (LBL)
  Bounds: Left=-151680.0, Right=151680.0, Bottom=-151680.0, Top=151680.0
  Size:   30336 x 30336 pixels
  CRS:    stere
  ✅ STATUS: Completely encompasses the DFSAR Radar Swath. Safe for warping.

NEW_AVG_ILLUM
  Bounds: Left=-151740.0, Right=151740.0, Bottom=-151740.0, Top=151740.0
  Size:   5058 x 5058 pixels
  CRS:    stere
  ✅ STATUS: Completely encompasses the DFSAR Radar Swath. Safe for warping.

NEW_PSR_MASK
  Bounds: Left=-151740.0, Right=151740.0, Bottom=-151740.0, Top=151740.0
  Size:   5058 x 5058 pixels
  CRS:    stere
  ✅ STATUS: Completely encompasses the DFSAR Rad